## Earnings-Return Relation

#### 1. Import the standard libraries:
 - pandas
 - numpy
 - datetime
 - statsmodels.formula.api

In [57]:
import pandas as pd
import numpy as np
import datetime as dt
import statsmodels.formula.api as sf

#### 2. Bring in Compustat Quarterly data
First, download Quarterly (NOT Annual) financial statement data from Compustat/CRSP merged database.

We will focus on the period 2011-01 to 2025-12 (fiscal) and the following data items:
- gvkey (firm ID).
- lpermno (CRSP identifier).
- linktype (indicator for link quality. LC and LU are desirable).
- rdq (quarterly earnings report date).
- fqtr (fiscal quarter = 1, 2, 3, or 4).
- ibq (quarterly income before extraordinary items).
- cshoq (common shares outstanding at the end of the quarter).
- prccq (common share price at the end of the quarter).
- sic (SIC code).
- atq (quarterly total assets).
- ceqq (quarterly common shareholders' equity).

Use Pandas methods read_csv() to import data into Python.
 - Save the data file and code file under the same folder.
 - Can add an argument, index_col=False, when a csv file is formatted poorly, and the read_csv method doesn't return meaningful results.
 - Can add an argument, usecols=['col1','col2','col3'], if we only want a subset of columns.

In [58]:
# function so i only type the file name (works on both computers) [I made cause I don't keep stuff in same folder]
def load_data(file_name, rows=5):
    # check the end of the file name to pick the right reader
    if file_name.endswith('.csv'):
        reader = pd.read_csv
    else:
        reader = pd.read_excel
    # try personal computer path first, then school desktop
    try:
        df = reader('C:\\Users\\Henry\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    except FileNotFoundError:
        df = reader('C:\\Users\\fbw4vt\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, index_col=False)
    # show the first rows (5 unless i give a different number)
    display(df.head(rows))
    return df

compustat = load_data('HW5.csv')

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
0,1004,LU,54594,2011-08-31,2011,1,INDL,C,D,STD,...,2011Q3,2011Q1,2011-09-15,1752.372,849.685,40.461,16.600,A,23.59,5080
1,1004,LU,54594,2011-11-30,2011,2,INDL,C,D,STD,...,2011Q4,2011Q2,2011-12-15,1821.612,863.948,40.284,17.581,A,18.26,5080
2,1004,LU,54594,2012-02-29,2011,3,INDL,C,D,STD,...,2012Q1,2011Q3,2012-03-20,2220.293,890.080,40.288,20.663,A,22.05,5080
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
4,1004,LU,54594,2012-08-31,2012,1,INDL,C,D,STD,...,2012Q3,2012Q1,2012-09-19,2170.300,880.600,39.953,18.200,A,14.88,5080


**Inspect the data**
 - Use the info() method to generate a summary
 - Are variable types appropriate? If not, use the astype() method, e.g., df['column_name'] = df['column_name'].astype(float), or to_datetime() method to convert to the date format.
 - Which variables have missing values?
 - Do variables look right?

In [59]:
compustat.info()

<class 'pandas.DataFrame'>
RangeIndex: 333603 entries, 0 to 333602
Data columns (total 21 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GVKEY     333603 non-null  int64  
 1   LINKTYPE  333603 non-null  str    
 2   LPERMNO   333603 non-null  int64  
 3   datadate  333603 non-null  str    
 4   fyearq    333603 non-null  int64  
 5   fqtr      333603 non-null  int64  
 6   indfmt    333603 non-null  str    
 7   consol    333603 non-null  str    
 8   popsrc    333603 non-null  str    
 9   datafmt   333603 non-null  str    
 10  curcdq    333603 non-null  str    
 11  datacqtr  333145 non-null  str    
 12  datafqtr  333603 non-null  str    
 13  rdq       300376 non-null  str    
 14  atq       300194 non-null  float64
 15  ceqq      299263 non-null  float64
 16  cshoq     327141 non-null  float64
 17  ibq       300790 non-null  float64
 18  costat    333603 non-null  str    
 19  prccq     313407 non-null  float64
 20  sic       33360

In [60]:
# change report date from text to a real date (to_datetime from the inspect instructions above)
compustat['rdq'] = pd.to_datetime(compustat['rdq'])

datacqtr (calendar year and quarter), rdq (earnings report date), atq (total assets), ceqq (common shareholders' equity), cshoq (common shares outstanding), ibq (income before extraordinary items), prccq (share price at quarter end) all have some null values (not 333603 non-null)

Only rdq needed changing (text to a date). The other variable types were fine.

#### 3. Remove rows
For this analysis, we will remove rows based on the following conditions:
- If rdq, cshoq, ibq, atq, prccq, or ceqq is missing.
- If total assets (atq) <= 10 (million).
- If the fiscal quarter (fqtr) is NOT 4 (i.e., let's focus on the last earnings announcement of the year).
- If the SIC code (sic) is between 6000 and 6999 (i.e., excluding the financial service firms).
- If link type (LINKTYPE) is NOT LC or LU (these indicate better link qualities).

In [61]:
# drop rows with missing values in the columns we're using
compustat = compustat.dropna(subset=['rdq','cshoq','ibq','atq','prccq',"ceqq"]) # from data cleaning lab step 6
compustat.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
0,1004,LU,54594,2011-08-31,2011,1,INDL,C,D,STD,...,2011Q3,2011Q1,2011-09-15,1752.372,849.685,40.461,16.600,A,23.59,5080
1,1004,LU,54594,2011-11-30,2011,2,INDL,C,D,STD,...,2011Q4,2011Q2,2011-12-15,1821.612,863.948,40.284,17.581,A,18.26,5080
2,1004,LU,54594,2012-02-29,2011,3,INDL,C,D,STD,...,2012Q1,2011Q3,2012-03-20,2220.293,890.080,40.288,20.663,A,22.05,5080
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
4,1004,LU,54594,2012-08-31,2012,1,INDL,C,D,STD,...,2012Q3,2012Q1,2012-09-19,2170.300,880.600,39.953,18.200,A,14.88,5080


In [62]:
# restrict to over 10mm assets
compustat = compustat[compustat['atq'] > 10]
compustat.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
0,1004,LU,54594,2011-08-31,2011,1,INDL,C,D,STD,...,2011Q3,2011Q1,2011-09-15,1752.372,849.685,40.461,16.600,A,23.59,5080
1,1004,LU,54594,2011-11-30,2011,2,INDL,C,D,STD,...,2011Q4,2011Q2,2011-12-15,1821.612,863.948,40.284,17.581,A,18.26,5080
2,1004,LU,54594,2012-02-29,2011,3,INDL,C,D,STD,...,2012Q1,2011Q3,2012-03-20,2220.293,890.080,40.288,20.663,A,22.05,5080
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
4,1004,LU,54594,2012-08-31,2012,1,INDL,C,D,STD,...,2012Q3,2012Q1,2012-09-19,2170.300,880.600,39.953,18.200,A,14.88,5080


In [63]:
# restrict to 4th quarter data
compustat = compustat[compustat['fqtr'] == 4]
compustat.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
7,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,2013Q2,2012Q4,2013-07-25,2136.900,918.600,39.382,0.600,A,20.06,5080
11,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,2014Q2,2013Q4,2014-07-15,2199.500,999.500,39.560,14.900,A,24.30,5080
15,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,2015Q2,2014Q4,2015-07-13,1515.000,845.100,35.423,-82.100,A,29.54,5080
19,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,2016Q2,2015Q4,2016-07-12,1442.100,865.800,34.515,11.800,A,24.41,5080


In [64]:
# exclude financial services firms
compustat = compustat[(compustat['sic'] < 6000) | (compustat['sic'] > 6999)] # e.g. keep values below 6000 and above 6999
compustat.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
7,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,2013Q2,2012Q4,2013-07-25,2136.900,918.600,39.382,0.600,A,20.06,5080
11,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,2014Q2,2013Q4,2014-07-15,2199.500,999.500,39.560,14.900,A,24.30,5080
15,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,2015Q2,2014Q4,2015-07-13,1515.000,845.100,35.423,-82.100,A,29.54,5080
19,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,2016Q2,2015Q4,2016-07-12,1442.100,865.800,34.515,11.800,A,24.41,5080


In [65]:
# keep only good quality links
compustat = compustat[compustat['LINKTYPE'].isin(['LC', 'LU'])] # using isin() function, not from the lab (same as HW 4)
compustat.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
7,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,2013Q2,2012Q4,2013-07-25,2136.900,918.600,39.382,0.600,A,20.06,5080
11,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,2014Q2,2013Q4,2014-07-15,2199.500,999.500,39.560,14.900,A,24.30,5080
15,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,2015Q2,2014Q4,2015-07-13,1515.000,845.100,35.423,-82.100,A,29.54,5080
19,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,2016Q2,2015Q4,2016-07-12,1442.100,865.800,34.515,11.800,A,24.41,5080


#### 4. Remove duplicates
Use drop_duplicates() method.

In [66]:
# remove rows that are exact duplicates
compustat = compustat.drop_duplicates()

# check the number of rows after cleaning (class reference was about 57,775)
print(compustat.shape)
compustat.head()

(57782, 21)


,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,datacqtr,datafqtr,rdq,atq,ceqq,cshoq,ibq,costat,prccq,sic
3,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,2012Q2,2011Q4,2012-07-17,2195.653,864.649,40.273,12.879,A,12.05,5080
7,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,2013Q2,2012Q4,2013-07-25,2136.900,918.600,39.382,0.600,A,20.06,5080
11,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,2014Q2,2013Q4,2014-07-15,2199.500,999.500,39.560,14.900,A,24.30,5080
15,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,2015Q2,2014Q4,2015-07-13,1515.000,845.100,35.423,-82.100,A,29.54,5080
19,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,2016Q2,2015Q4,2016-07-12,1442.100,865.800,34.515,11.800,A,24.41,5080


#### 5. Export data
We can use to_csv("file_name.txt", sep='\t', index = False, columns = ['col1','col2']). In particular, keep two columns in the output, LPERMNO and rdq.

In [67]:
# same idea as load_data, but saves to my Data_Files folder (works on both computers)
def save_data(df, file_name, columns):
    # try personal computer path first, then school desktop
    # (to_csv gives an OSError instead of FileNotFoundError when the folder doesn't exist)
    try:
        df.to_csv('C:\\Users\\Henry\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, sep='\t', index=False, columns=columns)
    except OSError:
        df.to_csv('C:\\Users\\fbw4vt\\OneDrive - University of Virginia\\COMM Semester 3\\1. Accounting Analytics\\Data_Files\\' + file_name, sep='\t', index=False, columns=columns)

# keep just the two columns the WRDS event study needs
save_data(compustat, 'earnings_events.txt', ['LPERMNO','rdq'])

#### 6. Use WRDS's Event Studies Module to calculate the abnormal return 
Steps will be shown in class.

I used the Canvas file in step 7 instead of running the event study, as recommended in class since it can take 10+ minutes.

#### 7. Import the abnormal return data
Can use the file from Canvas for now.

In [68]:
# bring in the abnormal return file from canvas
returns = load_data('eventus_edate.csv')

# change event date from text to a real date so it matches rdq for the merge
returns['evtdate'] = pd.to_datetime(returns['evtdate'])

,Model,permno,evtdate,cret,car,bhar,nrets_est
0,Fama-French Three Factor Model,10001,2013-04-01,0.044719,0.019580,0.019855,100
1,Fama-French Three Factor Model,10001,2012-04-10,-0.075103,-0.088903,-0.088402,100
2,Fama-French Three Factor Model,10001,2015-03-12,0.001001,0.015242,0.014467,100
3,Fama-French Three Factor Model,10001,2016-03-15,-0.097872,-0.108543,-0.111751,100
4,Fama-French Three Factor Model,10001,2017-03-16,0.001980,-0.218326,-0.243046,100


#### 8. Join the two datasets
Use merge() method. Specify four arguments: (1) the "left" DataFrame, (2) the "right" DataFrame, (3) the merge method, e.g., how='inner', and (4) on what variable(s), e.g., ['year'].
- 'inner' for inner join, 'left' for left outer join, 'right' for right outer join, and 'outer' for full outer join.
- if need to join on multiple variables (e.g., gvkey and year), then use on=['gvkey','year'].
- if the join on variables are of different names (e.g., cik and company_fkey), then use left_on='cik', right_on='company_fkey'.
- if need to join on multiple variables with different names, then use, e.g., left_on=['cik', 'year'], right_on=['company_fkey', 'fyear']. 

In [69]:
# join on firm and earnings date (the names are different in each file)
comp_ret = pd.merge(compustat, returns, how='inner', left_on=['LPERMNO','rdq'], right_on=['permno','evtdate'])

# check the number of rows after the merge
print(comp_ret.shape)
comp_ret.head()

(51980, 28)


,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,costat,prccq,sic,Model,permno,evtdate,cret,car,bhar,nrets_est
0,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,A,12.05,5080,Fama-French Three Factor Model,54594,2012-07-17,0.189227,0.211629,0.215263,100
1,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,A,20.06,5080,Fama-French Three Factor Model,54594,2013-07-25,0.205045,0.197778,0.214714,100
2,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,A,24.30,5080,Fama-French Three Factor Model,54594,2014-07-15,0.042735,0.071812,0.070660,100
3,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,A,29.54,5080,Fama-French Three Factor Model,54594,2015-07-13,-0.152229,-0.160396,-0.150531,100
4,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,A,24.41,5080,Fama-French Three Factor Model,54594,2016-07-12,0.180917,0.086685,0.088391,100


#### 9. Create variables
Now, let's create ROE, defined as ibq divided by ceqq.

In [70]:
# create roe (same way as dta = lt / at in the data cleaning lab)
comp_ret['roe'] = comp_ret['ibq'] / comp_ret['ceqq']
comp_ret.head()

,GVKEY,LINKTYPE,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,...,prccq,sic,Model,permno,evtdate,cret,car,bhar,nrets_est,roe
0,1004,LU,54594,2012-05-31,2011,4,INDL,C,D,STD,...,12.05,5080,Fama-French Three Factor Model,54594,2012-07-17,0.189227,0.211629,0.215263,100,0.014895
1,1004,LU,54594,2013-05-31,2012,4,INDL,C,D,STD,...,20.06,5080,Fama-French Three Factor Model,54594,2013-07-25,0.205045,0.197778,0.214714,100,0.000653
2,1004,LU,54594,2014-05-31,2013,4,INDL,C,D,STD,...,24.30,5080,Fama-French Three Factor Model,54594,2014-07-15,0.042735,0.071812,0.070660,100,0.014907
3,1004,LU,54594,2015-05-31,2014,4,INDL,C,D,STD,...,29.54,5080,Fama-French Three Factor Model,54594,2015-07-13,-0.152229,-0.160396,-0.150531,100,-0.097148
4,1004,LU,54594,2016-05-31,2015,4,INDL,C,D,STD,...,24.41,5080,Fama-French Three Factor Model,54594,2016-07-12,0.180917,0.086685,0.088391,100,0.013629


#### 10. Winsorizastion

In [71]:
# winsorization (same as data cleaning lab, just swapped in variables)
varlist = ['bhar','roe']
high=.99
low=.01
# replace values above the 99th percentile with the 99th percentile value
for x in varlist:
    comp_ret.loc[(comp_ret[x]>comp_ret[x].quantile(high, interpolation='nearest')),x]=comp_ret[x].quantile(high, interpolation='nearest')
# replace values below the 1st percentile with the 1st percentile value
for x in varlist:
    comp_ret.loc[(comp_ret[x]<comp_ret[x].quantile(low, interpolation='nearest')), x]=comp_ret[x].quantile(low, interpolation='nearest')

#### 11. Regression
Regress the buy-and-hold abnormal return (bhar) on earnings.

In [72]:
# linear regression, same 3 steps as the data analysis lab (roe is the earnings measure)
model1 = sf.ols(formula = 'bhar ~ roe', data=comp_ret) 
res1 = model1.fit() 
res1.summary() 

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   bhar   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     9.916
Date:                Wed, 04 Nov 2026   Prob (F-statistic):            0.00164
Time:                        23:50:37   Log-Likelihood:                 9099.3
No. Observations:               51962   AIC:                        -1.819e+04
Df Residuals:                   51960   BIC:                        -1.818e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0070      0.001     -7.763      0.000      -0.009      -0.005
roe            0.0077      0.002      3.149      0.002       0.003       0.013
==============================================================================
Omnibus:                     3648.206   Durbin-Watson:                   1.965
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            18254.354
Skew:                           0.113   Prob(JB):                         0.00
Kurtosis:                       5.895   Cond. No.                         2.75
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [ ]:
# additional calculation (not asked for in the instructions, not from the labs)
# I added this because the output above made it look like the effect size would be small
# effect size: how much bhar moves when roe goes up by one standard deviation

roe_coef = res1.params['roe']
roe_std = comp_ret['roe'].std()
effect_size = roe_coef * roe_std

print('coefficient on roe:', round(roe_coef, 4))
print('standard deviation of roe:', round(roe_std, 3))
print('effect size (change in bhar):', round(effect_size, 4))

coefficient on roe: 0.0077
standard deviation of roe: 0.364
effect size (change in bhar): 0.0028


#### 12. Discussion
Briefly summarize your findings. Is the association between earnings and abnormal return positive, negative, or statistically insignificant?

*The relationship between earnings and abnormal returns is significant and positive, with a positive coefficient for roe (.0077) and a p value of 0.002 (< .05). However it does not explain much, as the r-squared in the results shows 0.000 (about 0.0002), and the coefficient is so small that the effect size is only about 0.28 percentage points of bhar for a one standard deviation increase in roe (0.0077 x 0.364). The effect size is an additional calculation in the cell above, which I added because the output made it look like the effect would be small.*